## tl;dr
これは起動前の監査ノートブックです。実験条件の配布・データ整合性を検査し、モデルの性能改善はまだ評価しません。結果の正本はvalidation.jsonです。
## Context & Methods
### Key Assumptions
同一モデルの出発点は4条件で同一。真値はGit外。フォルダ分離だけではOS隔離になりません。旧input/evaluatorのmanifest、各候補コピー、資料の条件別配布、共通Python環境と価格付けテストを検査します。
## Data
PROTOCOL.mdとowner/validate_preparation.pyが定義・検査の出典です。生の非公開真値は表示しません。

In [1]:
from pathlib import Path
import json, subprocess
kit = Path('/Users/ankimo1210/Documents/projects/quant-agent-benchmark/experiments/research_library_round_02')
private = Path('/Users/ankimo1210/Documents/quant-agent-benchmark-private/research_library_round_02')
python_bin = private.parent / 'runtime-round-02-quantlib/bin/python'
assert kit.is_dir() and private.is_dir() and python_bin.exists()


## Results
### 1. 起動前の整合性検査
元提出物は読み取りのみ。validation.jsonだけを再生成します。候補の作業開始後にはこの起動前検査を使わないでください。

In [2]:
import os
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
run = subprocess.run([str(python_bin), str(kit/'owner/validate_preparation.py'), '--private-root', str(private), '--output', str(kit/'validation.json')], text=True, capture_output=True, env=env)
assert run.returncode == 0, run.stderr
result = json.loads((kit/'validation.json').read_text())
print(json.dumps({k: result[k] for k in ['preparation_valid', 'check_count', 'tests_run', 'tests_failed', 'launch_ready', 'models_started']}, indent=2))
assert result['preparation_valid'] and not result['launch_ready']


{
  "preparation_valid": true,
  "check_count": 260,
  "tests_run": 13,
  "tests_failed": 0,
  "launch_ready": false,
  "models_started": 0
}


## Takeaways
合格は実験資材の整合性を意味します。OS/ネットワーク隔離、実モデル設定の確認、実際のモデル起動・性能測定は別工程です。性能改善値・トークン・費用はまだ存在しません。